In [9]:
pip install split-folders

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("snmahsa/digit-dataset", output_dir="./dataset")

print("Path to dataset files:", path)



100%|██████████| 3.72M/3.72M [00:02<00:00, 1.43MB/s]

Extracting files...


Path to dataset files: ./dataset


In [ ]:
import splitfolders
import os
input_folder  = "dataset"       # 👈 change to your folder path
output_folder = "dataset_split" # output: train / val / test subfolders

splitfolders.ratio(
    input_folder,
    output=output_folder,
    seed=42,
    ratio=(0.8, 0.1, 0.1),   # train / val / test
    group_prefix=None,
    move=False                # set True to move instead of copy
)

print("✅ Split complete!")

Copying files: 2000 files [00:09, 207.72 files/s]

✅ Split complete!


In [13]:
for split in ["train", "val", "test"]:
    split_path = os.path.join(output_folder, split)
    if os.path.exists(split_path):
        classes = os.listdir(split_path)
        for cls in classes:
            count = len(os.listdir(os.path.join(split_path, cls)))
            print(f"{split}/{cls}: {count} files")

train/0: 160 files
train/1: 160 files
train/2: 160 files
train/3: 160 files
train/4: 160 files
train/5: 160 files
train/6: 160 files
train/7: 160 files
train/8: 160 files
train/9: 160 files
val/0: 20 files
val/1: 20 files
val/2: 20 files
val/3: 20 files
val/4: 20 files
val/5: 20 files
val/6: 20 files
val/7: 20 files
val/8: 20 files
val/9: 20 files
test/0: 20 files
test/1: 20 files
test/2: 20 files
test/3: 20 files
test/4: 20 files
test/5: 20 files
test/6: 20 files
test/7: 20 files
test/8: 20 files
test/9: 20 files


In [14]:
pip install scikit-learn opencv-python numpy tqdm


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import cv2
import pickle
import numpy as np
from tqdm import tqdm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score


dataset_dir = "./dataset_split"
IMG_SIZE    = (32, 32)
K           = 5
model_path  = "knn_model.pkl"

def load_images(split):
    x, y = [], []
    path = os.path.join(dataset_dir, split)
    for label in tqdm(sorted(os.listdir(path)), desc=split):
        label_path = os.path.join(path, label)
        for img_file in os.listdir(label_path):
            img = cv2.imread(os.path.join(label_path, img_file), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, IMG_SIZE).flatten()
            x.append(img)
            y.append(int(label))
    return np.array(x), np.array(y)

x_train, y_train = load_images('train')
print(f'train : {x_train.shape[0]} images')


knn = KNeighborsClassifier(n_neighbors=K, metric='euclidean', n_jobs=-1)
knn.fit(x_train, y_train)
print(f'model trained with K={K}')

with open(model_path, 'wb') as f:
    pickle.dump(knn, f)
print(f'model saved → {model_path}')

train: 100%|██████████| 10/10 [00:00<00:00, 12.58it/s]

train : 1600 images
model trained with K=5
model saved → knn_model.pkl


In [ ]:
def predict(image_path):
    with open(model_path, 'rb') as f:
        model = pickle.load(f)

    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)


    if img is None:
        print(f'❌ Could not open image: {image_path}')
        print(f'   Make sure the path is correct!')
        return None

    img = cv2.resize(img, IMG_SIZE).flatten().reshape(1, -1)

    result = model.predict(img)[0]
    print(f'predicted digit : {result}')
    return result
predict("./dataset_split/test/2/digit_2_105.png")

predicted digit : 2


np.int64(2)